In [ ]:
import pandas as pd
import numpy as np
from scipy.special import expit
from sklearn.model_selection import train_test_split

In [45]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import RandomForestClassifier

In [52]:
from sklearn.metrics import mean_absolute_error, r2_score

In [ ]:
# ============================
# 0. Setup
# ============================

np.random.seed(42)

df_original = pd.read_csv(
    "D:\\Projects\\ML\\Student Perfomance Prediction\\data\\raw\\test.csv"
)

print("Original Shape:", df_original.shape)

# Create Student_ID if missing
if 'Student_ID' not in df_original.columns:
    df_original['Student_ID'] = range(1, len(df_original) + 1)

# ============================
# 1. Cleanup
# ============================

df_original = df_original.dropna(subset=[
    'TestScore_Math','TestScore_Science','TestScore_Reading',
    'GPA','StudyHours','InternetAccess','Locale','AttendanceRate'
])

for col in [
    'TestScore_Math','TestScore_Science','TestScore_Reading',
    'GPA','StudyHours','InternetAccess','AttendanceRate'
]:
    df_original[col] = pd.to_numeric(df_original[col])

df_original['Locale'] = df_original['Locale'].astype(str).str.strip()

n_students = len(df_original)

# ============================
# 2. Longitudinal expansion
# ============================

df_expanded = df_original.loc[df_original.index.repeat(8)].reset_index(drop=True)

df_expanded['Semester_ID'] = np.tile(range(1,9), n_students)

# ============================
# 3. Static engineered features
# ============================

locale_map = {'Rural':3,'Town':2,'Suburban':2,'City':1}

df_expanded['Commute_Strain'] = df_expanded['Locale'].map(locale_map)

df_expanded['Assignment_Ratio'] = (
    (
        (df_expanded['StudyHours']*(df_expanded['InternetAccess']+1))/4
        * df_expanded['AttendanceRate']
    )
    * np.random.uniform(0.9,1.1,len(df_expanded))
    * 100
).clip(0,100).round(1)

# ============================
# 4. Internal Assessments
# ============================

# IA scores now generated from study behaviour NOT GPA

base_ia = (
    df_expanded['StudyHours']*0.3 +
    df_expanded['AttendanceRate']*0.5 +
    df_expanded['InternetAccess']*0.2
) * 10

ia1_noise = np.random.normal(0,2,len(df_expanded))
ia2_noise = np.random.normal(0,2,len(df_expanded))

df_expanded['IA1_Score'] = (base_ia + ia1_noise).clip(0,20).round()
df_expanded['IA2_Score'] = (base_ia + ia2_noise + 1).clip(0,20).round()

# ============================
# 5. GPA progression
# ============================

df_expanded['Prev_GPA'] = 0.0
df_expanded['Final_Sem_GPA'] = 0.0
df_expanded['Backlog_Count'] = 0

for sem in range(1,9):

    mask = df_expanded['Semester_ID'] == sem

    if sem == 1:

        # first semester previous GPA from dataset baseline
        df_expanded.loc[mask,'Prev_GPA'] = (
            df_expanded.loc[mask,'GPA']/4*10
        ).round(2)

        df_expanded.loc[mask,'Backlog_Count'] = 0

    else:

        prev_mask = df_expanded['Semester_ID'] == (sem-1)

        prev_gpa = df_expanded.loc[prev_mask,'Final_Sem_GPA'].values
        prev_backlogs = df_expanded.loc[prev_mask,'Backlog_Count'].values

        df_expanded.loc[mask,'Prev_GPA'] = prev_gpa

        # backlog generation
        fail_prob = np.clip((6-prev_gpa)/6,0.05,0.6)

        new_fails = np.random.binomial(2,fail_prob)
        cleared = np.random.binomial(prev_backlogs.astype(int),0.5)

        df_expanded.loc[mask,'Backlog_Count'] = np.clip(
            prev_backlogs + new_fails - cleared,
            0,
            15
        )

    # ============================
    # Final GPA generation
    # ============================

    prev_gpa_vals = df_expanded.loc[mask,'Prev_GPA'].values

    ia_effect = (
        df_expanded.loc[mask,'IA1_Score'].values*0.2 +
        df_expanded.loc[mask,'IA2_Score'].values*0.3
    )/20

    assignment_effect = (
        df_expanded.loc[mask,'Assignment_Ratio'].values/100
    )*0.4

    difficulty = np.random.uniform(-0.2,0.4,n_students)

    noise = np.random.normal(0,0.2,n_students)

    final_gpa = (
        prev_gpa_vals
        + ia_effect
        + assignment_effect
        - difficulty
        + noise
    )

    df_expanded.loc[mask,'Final_Sem_GPA'] = np.clip(final_gpa,0,10).round(2)

# ============================
# 6. Dropout probability
# ============================

gpa_penalty = (
    (5.5 - df_expanded['Final_Sem_GPA']).clip(lower=0)*0.08
)

risk_score = (
    df_expanded['Backlog_Count']*0.15 +
    df_expanded['Commute_Strain']*0.05 -
    df_expanded['Assignment_Ratio']*0.05 +
    df_expanded['Performance_Delta']*0.05 +
    gpa_penalty
)

df_expanded['Dropout_Probability'] = (
    expit(risk_score)*0.95
).round(2)

# binary dropout target
df_expanded['Dropout'] = (
    df_expanded['Dropout_Probability'] > 0.5
).astype(int)

# ============================
# 7. Output
# ============================

print("Final Expanded Shape:", df_expanded.shape)

columns_to_view = [
    'Student_ID',
    'Semester_ID',
    'Prev_GPA',
    'IA1_Score',
    'IA2_Score',
    'Final_Sem_GPA',
    'Backlog_Count',
    'Assignment_Ratio',
    'Dropout_Probability'
]

print("\n--- Sample Timeline for Student ID 1 ---")

print(
    df_expanded[df_expanded['Student_ID']==1][columns_to_view]
)

Original Shape: (999997, 21)
Final Expanded Shape: (7999976, 33)

--- Sample Timeline for Student ID 1 ---
   Student_ID  Semester_ID  Prev_GPA  IA1_Score  IA2_Score  Final_Sem_GPA  \
0           1            1      6.30        8.0        4.0           6.56   
1           1            2      6.56        0.0        5.0           6.02   
2           1            3      6.02        5.0        7.0           5.86   
3           1            4      5.86        6.0        9.0           5.72   
4           1            5      5.72        6.0        9.0           5.57   
5           1            6      5.57        6.0        7.0           5.23   
6           1            7      5.23        6.0        9.0           5.96   
7           1            8      5.96        7.0        5.0           6.19   

   Backlog_Count  Assignment_Ratio  Dropout_Probability  
0              0               6.6                 0.36  
1              0               7.3                 0.46  
2              0         

In [32]:
df_expanded.groupby("Semester_ID")["Dropout_Probability"].mean()

Semester_ID
1    0.147697
2    0.149674
3    0.150621
4    0.151018
5    0.151224
6    0.151283
7    0.151317
8    0.151299
Name: Dropout_Probability, dtype: float64

In [5]:
df_model=df_expanded.copy()

In [6]:
df_model.isna().sum()      

Age                    0
Grade                  0
Gender                 0
Race                   0
SES_Quartile           0
ParentalEducation      0
SchoolType             0
Locale                 0
TestScore_Math         0
TestScore_Reading      0
TestScore_Science      0
GPA                    0
AttendanceRate         0
StudyHours             0
InternetAccess         0
Extracurricular        0
PartTimeJob            0
ParentSupport          0
Romantic               0
FreeTime               0
GoOut                  0
Student_ID             0
Semester_ID            0
Commute_Strain         0
Assignment_Ratio       0
IA1_Score              0
IA2_Score              0
Performance_Delta      0
Prev_GPA               0
Final_Sem_GPA          0
Backlog_Count          0
Dropout_Probability    0
Dropout                0
dtype: int64

In [10]:
# Columns to stratify on
strat_cols = ['Locale', 'Gender', 'Race', 'SES_Quartile', 'ParentalEducation', 'SchoolType']

# Create a combined stratification column
df_model['strata'] = df_model[strat_cols].astype(str).agg('_'.join, axis=1)

# Perform stratified sampling: 300k rows
df_sample, _ = train_test_split(
    df_model,
    train_size=300_000,
    stratify=df_model['strata'],   # preserves proportions of all category combinations
    random_state=42
)

# Drop the temporary 'strata' column
df_sample = df_sample.drop(columns=['strata'])

# Quick checks
print(len(df_sample))  # 300000
print(df_sample['Locale'].value_counts())
print(df_sample['Gender'].value_counts())
print(df_sample['Race'].value_counts())
print(df_sample['SES_Quartile'].value_counts())
print(df_sample['ParentalEducation'].value_counts())
print(df_sample['SchoolType'].value_counts())  

300000
Locale
Suburban    117225
City         92838
Rural        56988
Town         32949
Name: count, dtype: int64
Gender
Female    153017
Male      146983
Name: count, dtype: int64
Race
White          131926
Hispanic        87107
Black           44907
Two-or-more     15023
Asian           14971
Other            6066
Name: count, dtype: int64
SES_Quartile
3    75058
2    74997
4    74988
1    74957
Name: count, dtype: int64
ParentalEducation
HS             97580
SomeCollege    90153
Bachelors+     56225
<HS            56042
Name: count, dtype: int64
SchoolType
Public     253342
Private     46658
Name: count, dtype: int64


In [ ]:
df_sample['Dropout_Flag']=(df_sample['Dropout_Probability']>=0.5).astype(int)
print(df_sample['Dropout_Flag'].value_counts())

Dropout_Flag
0    298712
1      1288
Name: count, dtype: int64


In [13]:
for col in df_sample.select_dtypes(include=['object']).columns:
    print(df_sample[col].unique())

['Male' 'Female']
['Asian' 'White' 'Hispanic' 'Other' 'Black' 'Two-or-more']
['Bachelors+' 'SomeCollege' 'HS' '<HS']
['Public' 'Private']
['Suburban' 'Rural' 'City' 'Town']


In [ ]:
def mapping_features(X):
  gender_map = {'Male': 0, 'Female': 1}
  X['Gender'] = X['Gender'].map(gender_map)
  education_map = {'<HS': 0, 'HS': 1, 'SomeCollege': 2, 'Bachelors+': 3}
  X['ParentalEducation'] = X['Pa rentalEducation'].map(education_map)
  X['SchoolType'] = X['SchoolType'].map({'Public': 0, 'Private': 1})
  X = pd.get_dummies(X, columns=['Locale'], drop_first=True)
  X = pd.get_dummies(X, columns=['Race'], drop_first=True) 
  return X
df_sample = mapping_features(df_sample)

In [ ]:
def feature_engineering(df):
  df['Academic_Performance'] = (df['TestScore_Math'] + df['TestScore_Science'] + 
                                df['TestScore_Reading']) / 3
  df['Engagement_Index'] = (df['AttendanceRate'] * 0.5 + df['StudyHours'] * 0.3 + 
                            df['Extracurricular'] * 0.2)
  df['Social_Distraction'] = (df['GoOut'] + df['FreeTime'] + df['Romantic'])
  df['Support_Score'] = (df['SES_Quartile'] + df['ParentalEducation'] + df['ParentSupport']) 
  df["Effort_Score"] = (df["StudyHours"] + df["AttendanceRate"] + df["ParentSupport"])
  df["IA_Average"] = (df["IA1_Score"] + df["IA2_Score"]) / 2
  df["IA_Improvement"] = (df["IA2_Score"] - df["IA1_Score"])
  df["IA_Consistency"] = abs(df_sample["IA2_Score"] - df["IA1_Score"])
  df['Performance_Delta'] = ((df['IA_Average'] / 2)- df['Prev_GPA'])
  df.drop(["TestScore_Math","TestScore_Reading","TestScore_Science"], axis=1, inplace=True)
  return df
df_sample = feature_engineering(df_sample)

In [16]:
df_sample = df_sample.sort_values(by=['Student_ID', 'Semester_ID'])
df_sample['Age_at_admission'] = df_sample.groupby('Student_ID')['Age'].transform('first')

In [17]:
df_sample["Academic_Year"] = ((df_sample["Semester_ID"] - 1) // 2) + 1
df_sample["Years_in_university"] = (df_sample["Semester_ID"] - 1) / 2

In [34]:
df_sample.columns

Index(['Age', 'Grade', 'Gender', 'SES_Quartile', 'ParentalEducation',
       'SchoolType', 'GPA', 'AttendanceRate', 'StudyHours', 'InternetAccess',
       'Extracurricular', 'PartTimeJob', 'ParentSupport', 'Romantic',
       'FreeTime', 'GoOut', 'Student_ID', 'Semester_ID', 'Commute_Strain',
       'Assignment_Ratio', 'IA1_Score', 'IA2_Score', 'Prev_GPA',
       'Final_Sem_GPA', 'Backlog_Count', 'Dropout_Probability', 'Dropout',
       'Dropout_Flag', 'Locale_Rural', 'Locale_Suburban', 'Locale_Town',
       'Race_Black', 'Race_Hispanic', 'Race_Other', 'Race_Two-or-more',
       'Race_White', 'Academic_Performance', 'Engagement_Index',
       'Social_Distraction', 'Support_Score', 'Effort_Score', 'IA_Average',
       'IA_Improvement', 'Age_at_admission', 'Academic_Year',
       'Years_in_university', 'IA_Consistency', 'Performance_Delta'],
      dtype='object')

In [37]:
static_features = ['Gender', 'SES_Quartile', 'ParentalEducation', 'SchoolType', 'InternetAccess', 'Extracurricular', 
                   'PartTimeJob', 'ParentSupport', 'Romantic', 'FreeTime', 'GoOut', 'Locale_Rural', 'Locale_Suburban',
                   'Locale_Town', 'Race_Black', 'Race_Hispanic', 'Race_Other', 'Race_Two-or-more', 'Race_White', 
                   'Age_at_admission']

historical_features = ['Prev_GPA', 'Backlog_Count', 'Academic_Year', 'Years_in_university']

current_behavioral_features = ['AttendanceRate', 'StudyHours', 'Commute_Strain', 'Assignment_Ratio', 'Engagement_Index',
                               'Social_Distraction', 'Support_Score', 'Effort_Score']

ia1_signal = ['IA1_Score']

ia2_signals = ['IA2_Score', 'IA_Average', 'IA_Improvement', 'Performance_Delta']

target_gpa = 'Final_Sem_GPA'
target_dropout = 'Dropout_Flag'

meta_columns = df_sample[['Student_ID','Semester_ID']]

def prepare_snapshot_datasets(df):
    X_before_ia1 = df[static_features + historical_features + current_behavioral_features]
    X_after_ia1 = df[static_features + historical_features + current_behavioral_features + ia1_signal]
    X_after_ia2 = df[static_features + historical_features + current_behavioral_features + ia1_signal + ia2_signals]
    y_gpa = df[target_gpa]
    y_dropout = df[target_dropout]
    return ( X_before_ia1, X_after_ia1, X_after_ia2, y_gpa, y_dropout)
df_before_ia, df_after_ia1, df_after_ia2, df_gpa, df_dropout = prepare_snapshot_datasets(df_sample)

In [39]:
students = df_sample['Student_ID'].unique()

train_students, test_students = train_test_split(students, test_size=0.2, random_state=42)

train_mask = df_sample['Student_ID'].isin(train_students)
test_mask = df_sample['Student_ID'].isin(test_students)

# BEFORE IA
X_before_train = df_before_ia[train_mask]
X_before_test = df_before_ia[test_mask]

# AFTER IA1
X_ia1_train = df_after_ia1[train_mask]
X_ia1_test = df_after_ia1[test_mask]

# AFTER IA2
X_ia2_train = df_after_ia2[train_mask]
X_ia2_test = df_after_ia2[test_mask]

# Targets
y_gpa_train = df_gpa[train_mask]
y_gpa_test = df_gpa[test_mask]

y_dropout_train = df_dropout[train_mask]
y_dropout_test = df_dropout[test_mask]

In [49]:
before_ia_rf_gpa = RandomForestRegressor(n_estimators=100, max_depth=10, n_jobs=-1, random_state=42)
before_ia_rf_gpa.fit(X_before_train, y_gpa_train)

RandomForestRegressor(max_depth=10, n_jobs=-1, random_state=42)

In [50]:
after_ia1_rf_gpa = RandomForestRegressor(n_estimators=100, max_depth=10, n_jobs=-1, random_state=42)
after_ia1_rf_gpa.fit(X_ia1_train, y_gpa_train)

RandomForestRegressor(max_depth=10, n_jobs=-1, random_state=42)

In [51]:
after_ia2_rf_gpa = RandomForestRegressor(n_estimators=100, max_depth=10, n_jobs=-1, random_state=42)
after_ia2_rf_gpa.fit(X_ia2_train, y_gpa_train)

RandomForestRegressor(max_depth=10, n_jobs=-1, random_state=42)

In [53]:
pred_before = before_ia_rf_gpa.predict(X_before_test)
pred_ia1 = after_ia1_rf_gpa.predict(X_ia1_test)
pred_ia2 = after_ia2_rf_gpa.predict(X_ia2_test)

print("Before IA")
print("MAE:", mean_absolute_error(y_gpa_test, pred_before))
print("R2:", r2_score(y_gpa_test, pred_before))

print("\nAfter IA1")
print("MAE:", mean_absolute_error(y_gpa_test, pred_ia1))
print("R2:", r2_score(y_gpa_test, pred_ia1))

print("\nAfter IA2")
print("MAE:", mean_absolute_error(y_gpa_test, pred_ia2))
print("R2:", r2_score(y_gpa_test, pred_ia2))

Before IA
MAE: 0.16749662521162062
R2: 0.9675581870507113

After IA1
MAE: 0.16718185451088255
R2: 0.9676849800447588

After IA2
MAE: 0.1663143876733065
R2: 0.9680578633048348


In [54]:
corr = df_sample.corr(numeric_only=True)
corr['Final_Sem_GPA'].sort_values(ascending=False).head(15)

Final_Sem_GPA           1.000000
Prev_GPA                0.978619
GPA                     0.784471
Academic_Performance    0.651434
AttendanceRate          0.602652
Assignment_Ratio        0.503365
StudyHours              0.475327
Engagement_Index        0.419045
IA_Average              0.383084
Years_in_university     0.374257
Semester_ID             0.374257
Academic_Year           0.365602
Effort_Score            0.318483
IA2_Score               0.312850
IA1_Score               0.308532
Name: Final_Sem_GPA, dtype: float64

In [55]:
feat_imp = pd.Series(before_ia_rf_gpa.feature_importances_, index=X_before_train.columns)
feat_imp.sort_values(ascending=False).head(10)

Prev_GPA               0.994418
Assignment_Ratio       0.003939
StudyHours             0.000395
AttendanceRate         0.000253
Engagement_Index       0.000247
Effort_Score           0.000222
Social_Distraction     0.000060
Support_Score          0.000054
Years_in_university    0.000053
FreeTime               0.000042
dtype: float64

In [67]:
X_copy=X_before_train.copy()
X_copy.drop(columns=['Prev_GPA'], inplace=True)

X_copy2=X_before_test.copy()
X_copy2.drop(columns=['Prev_GPA'], inplace=True)


In [70]:
before_ia_rf2_gpa = RandomForestRegressor(n_estimators=100, max_depth=10, n_jobs=-1, random_state=42)
before_ia_rf2_gpa.fit(X_copy, y_gpa_train)

pred_before2 = before_ia_rf2_gpa.predict(X_copy2)

print("Before IA")
print("MAE:", mean_absolute_error(y_gpa_test, pred_before2))
print("R2:", r2_score(y_gpa_test, pred_before2))

Before IA
MAE: 0.618203856913235
R2: 0.5958258696852055
